> English companion notebook for the radar FFT processing workflow. Code cells are kept aligned with the Chinese version.

# Radar FFT Cube Progress

This notebook explains the radar signal-processing path from DCA1000 raw ADC data to a range-Doppler-angle cube and then to candidate point-cloud detections.

The purpose is educational and engineering-oriented. Instead of treating FFT as a magic function, each stage is tied to the physical question it answers:

- Range FFT: how far is the reflector?
- Doppler FFT: how fast is it moving?
- Angle FFT: from which direction did it arrive?

This is the processing backbone behind the mmLock point-cloud pipeline.


## 1. Processing Pipeline Overview

The radar data path is:

1. Configure radar parameters: chirps, ADC samples, TX/RX layout, frame size.
2. Read the raw DCA1000 binary stream.
3. Reshape bytes into complex ADC samples.
4. Arrange samples by frame, chirp, receiver, and ADC index.
5. Separate TDM-MIMO transmitters into virtual antennas.
6. Apply Range FFT along fast-time samples.
7. Apply Doppler FFT across chirps.
8. Apply Angle FFT across the virtual antenna dimension.
9. Detect strong peaks and convert bins into physical points.

The output is not just a tensor. It is a physical interpretation of reflections around the radar.


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np

# Optional: only used for quickly inspecting the detection distribution.
# If matplotlib is unavailable, comment out the plotting cells below.
import matplotlib.pyplot as plt


## 2. Configure Capture Parameters

The configuration block defines how to interpret the raw binary file. Raw ADC data has no self-describing shape, so the script must know the radar setup.

Important parameters include:

- `num_rx`: number of receiver antennas.
- `num_tx`: number of transmitter antennas used in TDM-MIMO.
- `num_adc_samples`: samples collected inside one chirp.
- `num_chirps_per_tx`: chirps per transmitter per frame.
- `sample_rate`, `freq_slope`, and `start_freq`: parameters needed to convert bins into meters and meters per second.

If these values do not match the capture, every later FFT result will be physically wrong even if the code runs without errors.


In [ ]:
@dataclass
class RadarConfig:
    # Input file: replace this with the bin file exported by mmWave Studio/DCA1000.
    adc_bin_path: Path = Path("adc_data_Raw_0.bin")

    # Common IWR6843 configuration: 3 TX x 4 RX = 12 virtual antennas.
    num_tx: int = 3
    num_rx: int = 4

    # ADC samples per chirp. This must match the mmWave Studio profileCfg.
    num_adc_samples: int = 256

    # Number of chirp loops per TX in each frame. In TDM-MIMO, total chirps per frame are num_tx * num_loops_per_frame.
    num_loops_per_frame: int = 128

    # FFT size may be larger than the original sample count for zero padding.
    range_fft_size: int = 256
    doppler_fft_size: int = 128
    angle_fft_size: int = 64

    # The three physical parameters below convert FFT bins into meters, meters per second, and angles.
    # Replace them with the real values from the mmWave Studio profile/chirp configuration.
    sample_rate_hz: float = 5.0e6
    slope_hz_per_s: float = 60.0e12
    start_freq_hz: float = 60.0e9
    chirp_period_s: float = 60.0e-6

    # DCA1000/mmWave Studio commonly outputs complex ADC data.
    # If the capture is real-only, handle it separately; this workflow assumes complex samples.
    is_complex: bool = True


cfg = RadarConfig()
cfg


## 3. Read DCA1000 Raw ADC Bin

DCA1000 stores the captured ADC stream as interleaved integer samples. A typical mmWave capture uses I/Q pairs, so two real integers combine into one complex sample.

The read step must answer three questions:

- How are I and Q interleaved?
- How are RX channels interleaved?
- How many samples form one complete frame?

Once the stream is converted to complex values, the rest of the notebook can treat it as radar signal data rather than as a byte file.


In [ ]:
def read_dca1000_complex_bin(path: Path, num_rx: int, num_adc_samples: int) -> np.ndarray:
    """Read a common TI DCA1000 complex ADC bin file.

    Parameters
    ----------
    path:
        Path to the raw bin exported by mmWave Studio/DCA1000.
    num_rx:
        Number of enabled RX antennas.
    num_adc_samples:
        Number of ADC samples per chirp.

    Returns
    -------
    adc_cube:
        Complex array with shape [num_chirps_total, num_rx, num_adc_samples].

    Notes
    -----
    This follows the common DCA1000 complex-data pattern used by TI examples:
    int16 stream -> complex LVDS samples -> chirp/RX/sample cube.

    For complex data, every complex sample has I and Q components. The raw stream
    is often arranged in groups that TI's reference parser reconstructs as:
        sample_0 = raw[i]     + 1j * raw[i + 2]
        sample_1 = raw[i + 1] + 1j * raw[i + 3]
    This is why the conversion below steps through the int16 array in groups of 4.
    """
    raw = np.fromfile(path, dtype=np.int16)
    if raw.size == 0:
        raise ValueError(f"Empty ADC bin file: {path}")

    if raw.size % 4 != 0:
        raise ValueError(
            f"Raw int16 count {raw.size} is not divisible by 4. "
            "Check whether the file is truncated or uses a different capture format."
        )

    # Recover complex LVDS stream. This vectorized form is equivalent to the common
    # TI readDCA1000 loop that consumes raw[i:i+4] and produces two complex samples.
    raw4 = raw.reshape(-1, 4)
    complex_stream = np.empty(raw4.shape[0] * 2, dtype=np.complex64)
    complex_stream[0::2] = raw4[:, 0].astype(np.float32) + 1j * raw4[:, 2].astype(np.float32)
    complex_stream[1::2] = raw4[:, 1].astype(np.float32) + 1j * raw4[:, 3].astype(np.float32)

    samples_per_chirp_all_rx = num_rx * num_adc_samples
    if complex_stream.size % samples_per_chirp_all_rx != 0:
        raise ValueError(
            "Complex sample count does not fit num_rx * num_adc_samples. "
            f"complex_samples={complex_stream.size}, "
            f"num_rx={num_rx}, num_adc_samples={num_adc_samples}. "
            "Update RadarConfig to match the mmWave Studio capture settings."
        )

    num_chirps_total = complex_stream.size // samples_per_chirp_all_rx

    # First reshape into [chirp, rx * sample]. Inside each chirp, samples are grouped
    # by RX channel in the common TI parser output.
    lvds = complex_stream.reshape(num_chirps_total, samples_per_chirp_all_rx)

    # Final ADC cube: one chirp contains num_rx receive channels, and each RX channel
    # contains num_adc_samples fast-time ADC samples.
    adc_cube = lvds.reshape(num_chirps_total, num_rx, num_adc_samples)
    return adc_cube


## 4. Reshape into TDM-MIMO Frame Cube

TDM-MIMO means different TX antennas transmit at different chirp times. The receiver records all RX channels for each chirp, and the code later groups chirps by which TX produced them.

A useful mental model:

- TX is the sender of the chirp.
- RX is the listener receiving echoes.
- Each TX-RX pair forms one virtual antenna position.
- Combining TX-RX pairs creates the virtual array used for angle estimation.

The frame cube must preserve this organization. A wrong reshape can mix antennas or chirps, which destroys Doppler and angle information.


In [ ]:
def reshape_tdm_mimo_frames(adc_cube: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Reshape [total_chirp, rx, sample] into [frame, loop, tx, rx, sample]."""
    chirps_per_frame = cfg.num_tx * cfg.num_loops_per_frame
    num_complete_frames = adc_cube.shape[0] // chirps_per_frame

    if num_complete_frames == 0:
        raise ValueError(
            "Not enough chirps for one complete frame. "
            f"total_chirps={adc_cube.shape[0]}, chirps_per_frame={chirps_per_frame}."
        )

    dropped_chirps = adc_cube.shape[0] - num_complete_frames * chirps_per_frame
    if dropped_chirps:
        print(f"Dropping {dropped_chirps} incomplete chirps at the end of the file.")

    adc_cube = adc_cube[: num_complete_frames * chirps_per_frame]

    # Shape explanation:
    #   frame: capture frame index
    #   loop: repeated slow-time chirps for Doppler FFT
    #   tx: TDM-MIMO transmitter index
    #   rx: physical receiver index
    #   sample: fast-time ADC sample index for range FFT
    frame_cube = adc_cube.reshape(
        num_complete_frames,
        cfg.num_loops_per_frame,
        cfg.num_tx,
        cfg.num_rx,
        cfg.num_adc_samples,
    )
    return frame_cube


## 5. First FFT: Range FFT

Range FFT is applied along the ADC-sample axis inside each chirp. In FMCW radar, a target's distance becomes a beat frequency after mixing transmitted and received chirps.

The FFT separates those beat frequencies into range bins. A stronger magnitude at a range bin means there is likely reflected energy at that distance.

This step converts the raw waveform dimension into a distance-like dimension. It is the first reason FMCW radar can produce spatial sensing data rather than only signal strength.


In [ ]:
def range_fft(frame: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Run range FFT along the ADC sample axis.

    Parameters
    ----------
    frame:
        One frame with shape [loop, tx, rx, sample].

    Returns
    -------
    range_cube:
        Complex range cube with shape [loop, tx, rx, range_bin].
    """
    window = np.hanning(cfg.num_adc_samples).astype(np.float32)

    # Apply window on the fast-time ADC sample axis, then FFT into range bins.
    windowed = frame * window[None, None, None, :]
    cube = np.fft.fft(windowed, n=cfg.range_fft_size, axis=-1)

    # Only positive beat frequencies are physically useful for normal FMCW range.
    # Keeping all bins can be useful for debugging; here we keep the full FFT and
    # handle valid range bins during point extraction.
    return cube


## 6. Second FFT: Doppler FFT

Doppler FFT is applied across repeated chirps. If a reflector moves, the phase at a fixed range bin changes over chirps. The rate of that phase change corresponds to radial velocity.

This stage turns a sequence of chirps into velocity bins. For user-leaving detection, Doppler is critical because the target action is movement, not just presence.

After Range and Doppler FFT, each cell roughly means: reflected energy at this distance and this radial velocity.


In [ ]:
def doppler_fft(range_cube: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Run Doppler FFT along the slow-time loop axis."""
    window = np.hanning(cfg.num_loops_per_frame).astype(np.float32)

    # Slow-time window reduces Doppler sidelobes.
    windowed = range_cube * window[:, None, None, None]

    # FFT over loop axis. fftshift moves zero Doppler to the center.
    cube = np.fft.fft(windowed, n=cfg.doppler_fft_size, axis=0)
    cube = np.fft.fftshift(cube, axes=0)
    return cube


## 7. Third FFT: Angle FFT

Angle FFT is applied across the virtual antenna dimension. Reflections arriving from different directions produce different phase patterns across antennas.

With enough virtual antennas, the FFT can separate those phase patterns into angle bins. This gives the direction of arrival and allows the system to place points in a 2D or 3D coordinate system.

This step is why antenna arrangement matters. More useful spatial aperture usually gives better angular resolution.


In [ ]:
def angle_fft(doppler_cube: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Run angle FFT over the virtual antenna axis.

    Returns a complex cube with shape [doppler_bin, angle_bin, range_bin].
    """
    num_virtual_ant = cfg.num_tx * cfg.num_rx

    # Combine TX and RX into a virtual antenna dimension.
    # Input:  [doppler, tx, rx, range]
    # Output: [doppler, virtual_ant, range]
    virtual_cube = doppler_cube.reshape(
        cfg.doppler_fft_size,
        num_virtual_ant,
        cfg.range_fft_size,
    )

    # Window virtual array before angle FFT to reduce angular sidelobes.
    window = np.hanning(num_virtual_ant).astype(np.float32)
    windowed = virtual_cube * window[None, :, None]

    # FFT over virtual antenna axis. fftshift moves broadside near the center bin.
    cube = np.fft.fft(windowed, n=cfg.angle_fft_size, axis=1)
    cube = np.fft.fftshift(cube, axes=1)
    return cube


## 8. Convert FFT Bins to Physical Quantities

FFT output bins are indexes. To interpret them, the code converts bins into physical units:

- Range bin -> meters, based on bandwidth and sampling configuration.
- Doppler bin -> meters per second, based on chirp timing and carrier frequency.
- Angle bin -> direction, based on antenna spacing and wavelength.

This conversion is what makes the result readable as a point cloud. Without it, the model would receive arbitrary indexes that are harder to compare across captures.


In [ ]:
LIGHT_SPEED = 299_792_458.0


def range_axis_m(cfg: RadarConfig) -> np.ndarray:
    """Return range value in meters for each range FFT bin."""
    beat_freq = np.arange(cfg.range_fft_size) * cfg.sample_rate_hz / cfg.range_fft_size
    return LIGHT_SPEED * beat_freq / (2.0 * cfg.slope_hz_per_s)


def velocity_axis_mps(cfg: RadarConfig) -> np.ndarray:
    """Return velocity value in m/s for each shifted Doppler FFT bin."""
    wavelength = LIGHT_SPEED / cfg.start_freq_hz

    # Same-TX chirps are separated by num_tx chirp periods in TDM-MIMO.
    slow_time_step = cfg.num_tx * cfg.chirp_period_s
    doppler_freq = np.fft.fftshift(np.fft.fftfreq(cfg.doppler_fft_size, d=slow_time_step))
    return doppler_freq * wavelength / 2.0


def angle_axis_deg(cfg: RadarConfig) -> np.ndarray:
    """Return approximate azimuth angle for each shifted angle FFT bin.

    This assumes a uniform linear virtual array with lambda/2 spacing. For full
    IWR6843 azimuth/elevation processing, replace this with calibrated antenna geometry.
    """
    shifted_bins = np.arange(cfg.angle_fft_size) - cfg.angle_fft_size // 2
    sin_theta = 2.0 * shifted_bins / cfg.angle_fft_size
    sin_theta = np.clip(sin_theta, -1.0, 1.0)
    return np.degrees(np.arcsin(sin_theta))


ranges_m = range_axis_m(cfg)
velocities_mps = velocity_axis_mps(cfg)
angles_deg = angle_axis_deg(cfg)

print("range bins:", ranges_m[:5], "...")
print("velocity bins:", velocities_mps[:5], "...")
print("angle bins:", angles_deg[:5], "...")


## 9. Detect Points from the 3D FFT Cube

The FFT cube contains energy from humans, furniture, walls, multipath, and noise. Point detection selects the cells that look like meaningful reflectors.

A simple detector may threshold magnitude. A stronger implementation may use CFAR-style adaptive thresholds, neighborhood suppression, and region-of-interest filtering.

Each selected cell becomes a point with position, velocity, and intensity. This point set is the input expected by later point-cloud preprocessing and neural models.


In [ ]:
def detect_points_from_angle_cube(
    angle_cube: np.ndarray,
    cfg: RadarConfig,
    threshold_db_above_median: float = 18.0,
    max_points: int = 2048,
    min_range_m: float = 0.15,
    max_range_m: float | None = None,
) -> np.ndarray:
    """Detect point candidates from [doppler, angle, range] FFT cube.

    Returns a structured NumPy array with columns:
        range_m, velocity_mps, angle_deg, power_db, doppler_bin, angle_bin, range_bin
    """
    power = np.abs(angle_cube) ** 2
    power_db = 10.0 * np.log10(power + 1e-12)

    ranges = range_axis_m(cfg)
    velocities = velocity_axis_mps(cfg)
    angles = angle_axis_deg(cfg)

    valid_range_mask = ranges >= min_range_m
    if max_range_m is not None:
        valid_range_mask &= ranges <= max_range_m

    # Apply valid range mask by setting invalid bins to negative infinity.
    masked_power_db = power_db.copy()
    masked_power_db[:, :, ~valid_range_mask] = -np.inf

    finite_values = masked_power_db[np.isfinite(masked_power_db)]
    if finite_values.size == 0:
        raise ValueError("No valid range bins remain after range filtering.")

    noise_floor_db = np.median(finite_values)
    threshold_db = noise_floor_db + threshold_db_above_median

    detection_mask = masked_power_db > threshold_db
    coords = np.argwhere(detection_mask)

    dtype = [
        ("range_m", "f4"),
        ("velocity_mps", "f4"),
        ("angle_deg", "f4"),
        ("power_db", "f4"),
        ("doppler_bin", "i4"),
        ("angle_bin", "i4"),
        ("range_bin", "i4"),
    ]

    if coords.size == 0:
        return np.zeros(0, dtype=dtype)

    scores = masked_power_db[detection_mask]
    top_idx = np.argsort(scores)[::-1][:max_points]
    coords = coords[top_idx]
    scores = scores[top_idx]

    points = np.zeros(coords.shape[0], dtype=dtype)
    for i, (doppler_bin, angle_bin, range_bin) in enumerate(coords):
        points[i] = (
            ranges[range_bin],
            velocities[doppler_bin],
            angles[angle_bin],
            scores[i],
            doppler_bin,
            angle_bin,
            range_bin,
        )

    return points


## 10. Process One Frame End to End

The frame-level function combines the previous steps into one repeatable path:

raw frame -> reshape -> Range FFT -> Doppler FFT -> Angle FFT -> peak detection -> physical point list.

This function is useful because downstream models operate on frame sequences. If one frame's processing is inconsistent, the whole sequence becomes noisy.

When debugging, inspect intermediate outputs. A clean range profile and a plausible Doppler map are easier to diagnose than a final classifier error.


In [ ]:
def process_one_frame(cfg: RadarConfig, frame_index: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Read raw bin, run 3 FFTs, and return angle cube plus detected points."""
    if not cfg.adc_bin_path.exists():
        raise FileNotFoundError(
            f"ADC bin not found: {cfg.adc_bin_path}. "
            "Set cfg.adc_bin_path to the file exported by mmWave Studio/DCA1000."
        )

    # Step 1: raw int16 stream -> complex ADC cube [total_chirp, rx, sample]
    adc_cube = read_dca1000_complex_bin(
        cfg.adc_bin_path,
        num_rx=cfg.num_rx,
        num_adc_samples=cfg.num_adc_samples,
    )
    print("ADC cube:", adc_cube.shape, "[total_chirp, rx, sample]")

    # Step 2: ADC cube -> TDM-MIMO frame cube [frame, loop, tx, rx, sample]
    frame_cube = reshape_tdm_mimo_frames(adc_cube, cfg)
    print("Frame cube:", frame_cube.shape, "[frame, loop, tx, rx, sample]")

    if frame_index >= frame_cube.shape[0]:
        raise IndexError(f"frame_index={frame_index} but only {frame_cube.shape[0]} frames are available.")

    frame = frame_cube[frame_index]

    # Step 3: first FFT, ADC samples -> range bins
    r_cube = range_fft(frame, cfg)
    print("Range cube:", r_cube.shape, "[loop, tx, rx, range]")

    # Step 4: second FFT, slow-time loops -> Doppler bins
    rd_cube = doppler_fft(r_cube, cfg)
    print("Range-Doppler cube:", rd_cube.shape, "[doppler, tx, rx, range]")

    # Step 5: third FFT, virtual antenna -> angle bins
    rda_cube = angle_fft(rd_cube, cfg)
    print("Range-Doppler-Angle cube:", rda_cube.shape, "[doppler, angle, range]")

    # Step 6: threshold detection -> point cloud candidates
    points = detect_points_from_angle_cube(rda_cube, cfg)
    print("Detected points:", points.shape[0])
    return rda_cube, points


In [ ]:
# Usage:
# 1. Change cfg.adc_bin_path to the bin file exported by mmWave Studio/DCA1000.
# 2. Change the profile/chirp/frame parameters in RadarConfig to the real capture configuration.
# 3. Run the two lines below.
#
# cfg.adc_bin_path = Path("/path/to/adc_data_Raw_0.bin")
# angle_cube, points = process_one_frame(cfg, frame_index=0)


## 11. Simple Visualization

Visualization is not just for presentation. It is a sanity check for the signal-processing pipeline.

Useful plots include:

- range profile: are there clear peaks at plausible distances?
- range-Doppler map: does motion appear in expected velocity bins?
- angle spectrum: does the direction estimate change reasonably?
- point-cloud scatter plot: does the human body occupy a plausible region?

If these plots look random, the issue is likely before the model: configuration, reshape, calibration, or FFT axis selection.


In [ ]:
def plot_range_doppler(angle_cube: np.ndarray, cfg: RadarConfig) -> None:
    """Plot a range-Doppler heatmap by max-pooling over angle bins."""
    rd_power = np.max(np.abs(angle_cube) ** 2, axis=1)
    rd_power_db = 10.0 * np.log10(rd_power + 1e-12)

    ranges = range_axis_m(cfg)
    velocities = velocity_axis_mps(cfg)

    plt.figure(figsize=(10, 5))
    plt.imshow(
        rd_power_db,
        aspect="auto",
        origin="lower",
        extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
        cmap="viridis",
    )
    plt.xlabel("Range (m)")
    plt.ylabel("Velocity (m/s)")
    plt.title("Range-Doppler Heatmap")
    plt.colorbar(label="Power (dB)")
    plt.tight_layout()


def plot_points_angle_range(points: np.ndarray) -> None:
    """Plot detected points in angle-range space."""
    if points.size == 0:
        print("No points to plot.")
        return

    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(
        points["angle_deg"],
        points["range_m"],
        c=points["power_db"],
        s=8,
        cmap="magma",
    )
    plt.xlabel("Angle (deg)")
    plt.ylabel("Range (m)")
    plt.title("Detected Radar Points")
    plt.colorbar(scatter, label="Power (dB)")
    plt.tight_layout()


In [ ]:
# If process_one_frame has already produced angle_cube and points, uncomment the lines below:
# plot_range_doppler(angle_cube, cfg)
# plot_points_angle_range(points)


## 12. Engineering Extensions

This notebook is a readable baseline. A production-quality radar pipeline can add:

- calibration for antenna phase and amplitude mismatch;
- static clutter removal;
- CFAR detection instead of fixed thresholds;
- multi-frame tracking for target continuity;
- better point-cloud filtering and clustering;
- synchronization between radar frames and action labels.

These extensions matter for mmLock because the final security decision depends on stable motion features, not just on producing any point cloud.
